In [1]:
import os, math, json
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Trỏ vào output của notebook M3 (add as dataset input)
CHECKPOINT_PATH = "/kaggle/input/notebooks/hauutogpu/m3-gemma4-e2b-it/models/m3_gemma/checkpoints/checkpoint-1250/trainer_state.json"
LOGS_DIR    = "/kaggle/working/logs_m3"
FIGURES_DIR = "/kaggle/working/figures_m3"
os.makedirs(LOGS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# 1. Đọc trainer_state.json
with open(CHECKPOINT_PATH) as f:
    state = json.load(f)

log_df = pd.DataFrame(state["log_history"])
log_df.to_csv(f"{LOGS_DIR}/training_history.csv", index=False)
print(f"✅ Saved training_history.csv ({len(log_df)} rows)")

# 2. Vẽ loss curve
train_logs = log_df.dropna(subset=["loss"])
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_logs["step"], train_logs["loss"], color="royalblue", linewidth=1.5, label="Train Loss")
ax.set_xlabel("Step"); ax.set_ylabel("Loss")
ax.set_title("M3 — Gemma4 E2B QLoRA: Training Loss Curve")
ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(f"{FIGURES_DIR}/loss_curve.png", dpi=150)
print("✅ Saved loss_curve.png")

✅ Saved training_history.csv (125 rows)
✅ Saved loss_curve.png


In [2]:
!pip install -q -U bitsandbytes>=0.46.1
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q -U accelerate peft

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 111.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 37.0 MB/s eta 0:00:00


In [3]:
import math, torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from peft import PeftModel
from datasets import Dataset

# Paths
M3_OUTPUT = "/kaggle/input/notebooks/hauutogpu/m3-gemma4-e2b-it"
BASE_MODEL_ID = "google/gemma-4-E2B-it"
FINAL_MODEL   = f"{M3_OUTPUT}/models/m3_gemma/final"
# Load test set (đã được lưu trong output M3)
test_df = pd.read_csv(f"{M3_OUTPUT}/models/m3_gemma/test_set_random.csv")
test_dataset = Dataset.from_pandas(test_df)

# Load tokenizer + model từ final/
tokenizer = AutoTokenizer.from_pretrained(FINAL_MODEL)
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
                                 bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, quantization_config=bnb_config, device_map="auto")
model = PeftModel.from_pretrained(base_model, FINAL_MODEL)
model.eval()

# Tokenize
def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=256)

tokenized_test = test_dataset.map(tokenize_fn, batched=True, remove_columns=test_dataset.column_names)
data_collator  = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Evaluate
trainer = Trainer(model=model, 
                  data_collator=data_collator,
                  args=TrainingArguments(
                      output_dir="/kaggle/working", 
                      report_to="none",
                      per_device_eval_batch_size=1,
                      dataloader_pin_memory=False,
                      
                                        )
                 )
results = trainer.evaluate(eval_dataset=tokenized_test)
results["perplexity"] = math.exp(results["eval_loss"])

print(f"Test Loss  : {results['eval_loss']:.4f}")
print(f"Perplexity : {results['perplexity']:.4f}")

pd.DataFrame([results]).to_csv("/kaggle/working/test_metrics_m3.csv", index=False)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Map:   0%|          | 0/4826 [00:00<?, ? examples/s]

Training Loss,Validation Loss,Step
No log,0.605998,0


Test Loss  : 0.6060
Perplexity : 1.8331
